In [ ]:
import os
import glob
import json
import random
import numpy as np
import cv2
from datetime import datetime

from ai_edge_litert.interpreter import Interpreter

TFLITE_MODEL_PATH = "models/unet_food_int8_256.tflite"
interpreter = Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
img_size = (256, 256)
THRESHOLD=0.5

# def get_tablewaremask(images, unet_flg=False):
#     h, w, _ = images[0].shape
#     if unet_flg:
#         masks = []
#         for image in images:
#             image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#             image = cv2.resize(image, img_size).astype(np.float32) / 255.0
#             image = np.expand_dims(image, axis=0)
#             interpreter.set_tensor(input_details[0]['index'], image)
#             interpreter.invoke()
#             mask = interpreter.get_tensor(output_details[0]['index'])[0]
#             masks.append(mask[:,:,0] > THRESHOLD)
#         mask = np.any(np.stack(masks, axis=0), axis=0).astype(np.uint8) * 255
#         mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
#         masks = [cv2.resize(m.astype(np.uint8)*255, (w, h), interpolation=cv2.INTER_NEAREST) for m in masks]
#     else:
#         center = (w // 2, h // 2)
#         radius = int(min(h, w) / 2 * 0.5)
#         mask_circle = np.zeros((h, w), dtype=np.uint8)
#         mask = cv2.circle(mask_circle, center, radius, 255, -1)
#         masks = [mask] * 3
#     return mask, masks


def get_tablewaremask(images, unet_flg=False):
    h, w, _ = images[0].shape
    
    if unet_flg:
        masks = []
        for image in images:
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image_resized = cv2.resize(image_rgb, img_size).astype(np.float32) / 255.0
            image_dims = np.expand_dims(image_resized, axis=0)
            
            interpreter.set_tensor(input_details[0]['index'], image_dims)
            interpreter.invoke()
            
            mask = interpreter.get_tensor(output_details[0]['index'])[0]
            masks.append(mask[:,:,0] > THRESHOLD)
            
        # 1. 論理和マスクの作成とリサイズ
        mask = np.any(np.stack(masks, axis=0), axis=0).astype(np.uint8) * 255
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
        
        # 2. 各マスク画像のリサイズ（3次元化して保存用にするため、ここでは元の変数と分けるか再代入します）
        masks_resized = [cv2.resize(m.astype(np.uint8)*255, (w, h), interpolation=cv2.INTER_NEAREST) for m in masks]
        
        # --- 画像結合・保存処理の追加 ---
        # カラー画像を横に結合
        images_resized = [cv2.resize(img, (w, h)) for img in images]
        top_row = np.hstack(images_resized)
        
        # マスク画像を3チャンネル(BGR)に変換して横に結合
        masks_bgr = [cv2.cvtColor(m, cv2.COLOR_GRAY2BGR) for m in masks_resized]
        bottom_row = np.hstack(masks_bgr)
        
        # 上段と下段を縦に結合
        left_block = np.vstack([top_row, bottom_row])
        
        # 論理和マスクを3チャンネルに変換し、左ブロックの縦幅（2 * h）に合わせてリサイズ
        mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
        right_block = cv2.resize(mask_bgr, (int(w * (2 * h / h)), 2 * h), interpolation=cv2.INTER_NEAREST)
        
        # 左ブロックと右ブロックを横に結合
        combined_img = np.hstack([left_block, right_block])
        
        # 現在時刻のファイル名（例: 20260708_231700.png）で保存
        filename = datetime.now().strftime("%Y%m%d_%H%M%S.png")
        cv2.imwrite(f"mask/{filename}", combined_img)
        # ---------------------------------
        
        # 元の戻り値の形式に合わせる
        masks = masks_resized
        
    else:
        center = (w // 2, h // 2)
        radius = int(min(h, w) / 2 * 0.5)
        mask_circle = np.zeros((h, w), dtype=np.uint8)
        mask = cv2.circle(mask_circle, center, radius, 255, -1)
        masks = [mask] * 3
        
    return mask, masks

def calc_amount(depth_empty, depth_full, depth_after, meal_mask,
                depth_min=2000, depth_max=3500,
                inpaint_radius=3):

    # --- サイズ整合 ---
    h, w = depth_empty.shape
    depth_full  = cv2.resize(depth_full,  (w, h), interpolation=cv2.INTER_NEAREST)
    depth_after = cv2.resize(depth_after, (w, h), interpolation=cv2.INTER_NEAREST)
    
    
    # # 深度補正対応
    # depth_empty  = depth_empty*1000
    # depth_full   = depth_full*1000
    # depth_after  = depth_after*1000

    # --- 欠損補完処理 ---
    def clip_and_inpaint(depth):
        depth = depth.astype(np.float32)
        
        # 有効範囲外をNaNに
        depth[(depth < depth_min) | (depth > depth_max)] = np.nan

        # NaNマスク
        mask = np.isnan(depth).astype(np.uint8) * 255
        depth_filled = np.nan_to_num(depth, nan=0.0)

        # inpaint補完（欠損がある場合のみ）
        if np.any(mask):
            depth_inpainted = cv2.inpaint(depth_filled.astype(np.float32), mask, inpaint_radius, cv2.INPAINT_TELEA)
        else:
            depth_inpainted = depth_filled

        # 円マスク外をゼロ化（深度差計算対象外）
        depth_inpainted[meal_mask == 0] = 0

        return depth_inpainted

    # --- 各深度マップを補完＋マスク ---
    depth_empty = clip_and_inpaint(depth_empty)
    depth_full  = clip_and_inpaint(depth_full)
    depth_after = clip_and_inpaint(depth_after)

    # --- 深度差（高さマップ）の計算 ---
    # 満杯時と食後の深度を空皿基準で正規化
    h_full  = depth_empty - depth_full   # 満杯時の食材の高さ
    h_after = depth_empty - depth_after  # 食後の残存食材の高さ

    # --- 負値クリップ（物理的にありえない値を除外）---
    # h_full  = np.maximum(h_full, 0)
    # h_after = np.maximum(h_after, 0)

    # --- マスク内の有効ピクセル数を確認 ---
    valid_mask = meal_mask > 0
    n_valid = np.sum(valid_mask)
    
    if n_valid == 0:
        print("警告: 有効なマスク領域がありません")
        # return 0.0
        return None
    
    # --- 相対体積計算（マスク内のみ） ---
    # V_full  = np.sum(h_full[valid_mask])
    # V_after = np.sum(h_after[valid_mask])
    V_full  = np.nanmean(h_full[valid_mask])
    V_after = np.nanmean(h_after[valid_mask])

    # --- 残存率の計算 ---
    if V_full < 1e-6:  # ほぼゼロの場合
        # print("警告: 満杯時の体積がほぼゼロです")
        return 0.0
        # return None
    
    remaining_ratio = V_after / V_full

    remaining_ratio = float(np.clip(remaining_ratio, 0, 1))
    
    
    
    if V_full < 0:
        remaining_ratio = 1.0
    if V_after < 0:
        remaining_ratio = 0.0

    return remaining_ratio


# def view_result(y_true, y_pred, plate_id, plate_name):
#     import matplotlib.pyplot as plt
#     from sklearn.metrics import r2_score, confusion_matrix, mean_absolute_error
#     import seaborn as sns
    
#     print(f"Results for {plate_name}:")
#     r2 = r2_score(y_true, y_pred)
#     mae = mean_absolute_error(y_true, y_pred)
#     print(f"R² Score for {plate_name}: {r2:.4f}")
#     print(f"MAE for {plate_name}: {mae:.4f}")

#     levels = np.array([0, 0.2, 0.4, 0.6, 0.8, 1.0])

#     # --- インデックス化（これが重要） ---
#     def to_index(values, levels):
#         return np.array([np.argmin(np.abs(levels - v)) for v in values])

#     y_true_idx = to_index(y_true, levels)
#     y_pred_idx = to_index(y_pred, levels)

#     # --- 混同行列 ---
#     cm = confusion_matrix(y_true_idx, y_pred_idx)

#     # --- 可視化 ---
#     plt.figure()
#     sns.heatmap(
#         cm,
#         annot=True,
#         fmt="d",
#         xticklabels=levels,
#         yticklabels=levels,
#         cmap="Blues"
#     )

#     plt.title(f"Confusion Matrix for {plate_id}_{plate_name} (R²={r2:.4f}, MAE={mae:.4f})")
#     plt.xlabel("Predicted")
#     plt.ylabel("True")
#     plt.tight_layout()
#     plt.show()


def view_result(y_true, y_pred, plate_id, plate_name, r2, mae, save_path=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    from datetime import datetime

    levels = np.array([0, 0.2, 0.4, 0.6, 0.8, 1.0])

    def to_index(values, levels):
        return np.array([np.argmin(np.abs(levels - v)) for v in values])

    y_true_idx = to_index(y_true, levels)
    y_pred_idx = to_index(y_pred, levels)

    # 軸入れ替え（縦=Predicted, 横=True）
    cm = confusion_matrix(y_true_idx, y_pred_idx).T

    # 縦軸を下→上が昇順になるよう反転
    cm = np.flipud(cm)

    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=levels,
        yticklabels=levels[::-1],
        cmap="Blues"
    )

    plt.title(
        f"Confusion Matrix for {plate_id}_{plate_name}\n"
        f"N={len(y_true)}, R²={r2:.4f}, MAE={mae:.4f}"
    )

    plt.xlabel("True")
    plt.ylabel("Predicted")

    plt.tight_layout()

    if save_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = (
            f"result_analyze/"
            f"cm_{plate_id}_{plate_name}_{timestamp}.png"
        )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {save_path}")

    plt.show()
    plt.close()

def calc_score(y_true, y_pred):
    from sklearn.metrics import r2_score, mean_absolute_error
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return float(r2), float(mae)

def show_image_and_depth(image, depth):
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    ax1.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax1.axis('off')
    ax1.set_title('Image')
    im2 = ax2.imshow(depth, cmap='viridis')
    ax2.axis('off')
    ax2.set_title('Depth')
    plt.colorbar(im2, ax=ax2)
    plt.show()
    
    

def label_to_image(image, label_text):
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1.0
    thickness = 2

    # テキストサイズ取得
    (text_w, text_h), baseline = cv2.getTextSize(
        label_text, font, font_scale, thickness
    )

    # 上部余白サイズ
    margin_top = text_h + baseline + 20

    h, w = image.shape[:2]

    # 白背景の新画像を作成
    result = np.full(
        (h + margin_top, w, 3),
        255,
        dtype=np.uint8
    )

    # 元画像を貼り付け
    result[margin_top:, :] = image

    # テキスト描画位置
    x = max(10, (w - text_w) // 2)
    y = text_h + 10

    cv2.putText(
        result,
        label_text,
        (x, y),
        font,
        font_scale,
        (0, 0, 0),
        thickness,
        cv2.LINE_AA
    )

    return result


tmp_data = {}

for _ in range(100):
    print(_)

    plate_ids = [os.path.basename(p) for p in glob.glob("plate_images/*")]
    for plate_id in plate_ids:
        base_config = json.load(open(f"test_data/{plate_id}/plate_setting.json"))
        plate_names = [d["plate_name"] for d in base_config]
        nansai_flags = [d["nansai"] for d in base_config]
        full_plate_images = []
        for plate_name, nansai_flag in zip(plate_names, nansai_flags):
            if plate_name != "shusai" and plate_id != "B":
                continue
            dict_path = f"plate_images/{plate_id}/{plate_name}/"
            full_plate_path = random.choice(glob.glob(f"{dict_path}/plate_10_*.png"))
            full_plate_image = cv2.imread(full_plate_path)
            full_plate_depth = np.load(full_plate_path.replace(".png", ".npy"))
            empty_plate_path = random.choice(glob.glob(f"{dict_path}/plate_00_*.png"))
            empty_plate_image = cv2.imread(empty_plate_path)
            empty_plate_depth = np.load(empty_plate_path.replace(".png", ".npy"))
            target_plate_paths = glob.glob(f"{dict_path}/plate_*.png")
            # show_image_and_depth(empty_plate_image, empty_plate_depth)
            # show_image_and_depth(full_plate_image, full_plate_depth)
            
            full_plate_images.append(label_to_image(full_plate_image, plate_name))

            y_true, y_pred = [], []
            for target_plate_path in target_plate_paths:
                
                target_plate_image = cv2.imread(target_plate_path)
                target_plate_depth = np.load(target_plate_path.replace(".png", ".npy"))
                real_amount = float(os.path.basename(target_plate_path).split("_")[1])
                if real_amount != 8:
                    continue
                # 食器画像，食材マスク画像取得
                # print("a")
                tableware_images = [empty_plate_image, full_plate_image, target_plate_image]
                meal_mask, meal_masks = get_tablewaremask(tableware_images, unet_flg = not nansai_flag)
                # meal_mask, meal_masks = get_tablewaremask(tableware_images, unet_flg = False)
                # print("b")
                # 残存率を計算（0=全て食べた、1=全て残っている）
                remaining_ratio = calc_amount(empty_plate_depth, full_plate_depth, target_plate_depth, meal_mask)
                # if remaining_ratio == 1 and real_amount == 0:
                #     print(full_plate_path, empty_plate_path, target_plate_path)
                # print("c")
                if remaining_ratio is None:
                    print(f"スキップ: {plate_id} - {plate_name}")
                    continue
                
                y_true.append(real_amount / 10)  # 0.0 ~ 1.0 に正規化
                y_pred.append(remaining_ratio)
            
            try:
                # r2, mae = view_result(y_true, y_pred, plate_id, plate_name)
                r2, mae = calc_score(y_true, y_pred)
                
                key = f"{plate_id}_{plate_name}"
                if key not in tmp_data.keys():
                    tmp_data[key] = []
                tmp_data[key].append({
                    "plate_id": plate_id,
                    "plate_name": plate_name,
                    "y_true": y_true,
                    "y_pred": y_pred,
                    "r2": r2,
                    "mae": mae
                })

                # save_full_plate_image = cv2.vconcat([cv2.resize(img, (full_plate_images[0].shape[1], full_plate_images[0].shape[0])) for img in full_plate_images])
                # cv2.imwrite(f"results/{plate_id}_{plate_name}_full_plate_{datetime.now():%Y%m%d_%H%M%S}.png", save_full_plate_image)
            except Exception as e:
                print(e)
                print(f"結果の表示に失敗: {plate_id} - {plate_name}")

# tmp_dataから各行のr2のうち最大のものを抽出
final_results = []
for key, records in tmp_data.items():
    best_record = max(records, key=lambda x: x["r2"])
    final_results.append(best_record)
# jsonに保存
with open("final_results.json", "w") as f:
    json.dump(final_results, f, indent=4)

# tmp_dataから各行のr2のうち中央のものを抽出
final_results_median = []
for key, records in tmp_data.items():
    sorted_records = sorted(records, key=lambda x: x["r2"])
    median_record = sorted_records[len(sorted_records) // 2]
    final_results_median.append(median_record)
# jsonに保存
with open("final_results_median.json", "w") as f:
    json.dump(final_results_median, f, indent=4)

In [ ]:
# final_results.jsonからplate_id,plate_name, r2, maeを取り出しJSONで保存
with open("final_results.json", "r") as f:
    final_results = json.load(f)
for i in range(len(final_results)):
    final_results[i].pop("y_true")
    final_results[i].pop("y_pred")
with open("final_results_data.json", "w") as f:
    json.dump(final_results, f, indent=4)

In [ ]:
# jsonから読み込んでグラフを保存
with open("final_results.json", "r") as f:
    final_results = json.load(f)
for result in final_results:
    plate_id = result["plate_id"]
    plate_name = result["plate_name"]
    r2 = result["r2"]
    mae = result["mae"]
    
    # グラフを保存
    save_path = f"final_results/cm_{plate_id}_{plate_name}.png"
    view_result(result["y_true"], result["y_pred"], plate_id, plate_name, r2, mae, save_path=save_path)

# jsonから読み込んでグラフを保存
with open("final_results_median.json", "r") as f:
    final_results_median = json.load(f)
for result in final_results_median:
    plate_id = result["plate_id"]
    plate_name = result["plate_name"]
    r2 = result["r2"]
    mae = result["mae"]
    
    # グラフを保存
    save_path = f"final_results_median/cm_{plate_id}_{plate_name}.png"
    view_result(result["y_true"], result["y_pred"], plate_id, plate_name, r2, mae, save_path=save_path)